---

Experiments

---

In [ ]:
# autoload
%load_ext autoreload
%autoreload 2

# import necessary libraries
import numpy as np

# Load PGL libraries and start a PGL window
from pgl import pgl, pglTask, pglExperiment, pglParameter, pglParameterBlock, pglParameterNestedBlock, pglParameterBatch
from pgl import pglVWFATask
pgl = pgl()

# close any existing windows
pgl.cleanUp()

---

Random dot kinetogram task

---

In [ ]:
# Set up random dot task
class pglRandomDotTask(pglTask):
    
    #----------------------
    # initialize the task
    #----------------------
    def __init__(self, pgl):
        
        # initialize parent class
        super().__init__(pgl)
        
        # set task parameters, these will automatically be saved in the settings file
        self.settings.taskName = "Random Dot Motion Task"
        self.settings.seglen = [1, 0.5]
        
        # configuration
        self.settings.config.width = 15
        self.settings.config.height = 10
        self.settings.config.coherence = (0.1,1)
        self.settings.config.dir = np.arange(0,360,45)

        # add parameters for coherence and direction
        coherence = pglParameter('coherence',self.settings.config.coherence)
        dir = pglParameter('dir',self.settings.config.dir)
        
        # nested block makes sure that each combination of direction and coherence
        # appear once (in random order each block)
        self.addParameter(pglParameterNestedBlock([dir, coherence]))
        
        # initalize stimulus
        self.rdk = pgl.randomDots(width=self.settings.config.width, height=self.settings.config.height)
    def configure(self, e):
        e.printLastRuns(self)
    
    #----------------------
    # update screen function: gets called every screen refresh
    #----------------------
    def updateScreen(self):
        '''
        What to draw on each screen update
        '''
        # for the first segment
        if self.state.currentSegment==0:
            # display the random dot kinetogram, with the current parameters for direction and coherence
            self.rdk.display(direction=self.currentParams['dir'], coherence=self.currentParams['coherence'], speed=5.0)
        self.pgl.fixationCross()

---

Run RDK experiment

---

In [ ]:
# initialize experiment
e = pglExperiment(pgl=pgl,experimentName='randomDots',subjectID='s0000')

# add random dot task to the experiment
randomDataTask = pglRandomDotTask(pgl)
e.addTask(randomDataTask)

# initialize the screen and run
e.initScreen()
e.run()
e.display()

---

Print experiment run details

---

In [ ]:
e.print()

---

Annotated task for writing a new task

---

In [ ]:
# places that says "CHANGEME" need to be changed!!
class pgl_CHANGEME_Task(pglTask):
    
    ########################
    def __init__(self, pgl):
        super().__init__(pgl)
        
        # set task parameters, these will automatically be saved in the settings file
        self.settings.taskName = "CHANGEME"
        
        # how many trials to run for (set to np.inf if you want to manually stop experiment)
        self.settings.nTrials = np.inf                      #CHANGEME
        
        # set seglens in seconds, in this case we have 3 segments
        # one at the beginning for fixation, another to display the stimulus
        # and a final segment for response
        self.settings.seglen = [0.5, 1, 1]                  #CHANGEME

        # configuration parameters, these will automatically be saved in the settings file
        # they can be anything that you want/need
        self.settings.config.coherence = (0.2,0.6,1)        #CHANGEME
        self.settings.config.dir = (0, 180)                 #CHANGEME
        self.settings.config.stimulusEccentricity = 10      #CHANGEME
        self.settings.config.stimulusSize = 10              #CHANGEME
        self.settings.config.speed = 5                      #CHANGEME
        
        # add parameters for coherence and direction
        coherence = pglParameter('coherence',self.settings.config.coherence) #CHANGEME
        dir = pglParameter('dir',self.settings.config.dir)                   #CHANGEME
        
        # nested block makes sure that each combination of direction and coherence
        # appear once (in random order each block)
        self.addParameter(pglParameterNestedBlock([dir, coherence]))         #CHANGEME
        
        # initalize stimulus
        self.rdk = pgl.randomDots(width=self.settings.config.stimulusSize, height=self.settings.config.stimulusSize)                
                        
    ########################
    def startSegment(self, startTime):
        '''
        Start a segment. This is where you write code
            for an processing at the beginning of a segment
            for example, setting up a stimulus for the segment
            that will be displayed in updateScreen
        '''
        super().startSegment(startTime)
    
        # CHANGEME - this is how you detect which segment we are on
        if self.state.currentSegment == 0: 
            # Here, we are just resetting whether we have a response or not
            # note that this is set in the state: the variable which contains
            # changing states as opposed to settings and data
            self.state.gotResponse = False
            # also setting the fixationColor which is used in updateScreen #CHANGEME
            self.state.fixationColor = 1
    
    ########################
    # updateScren
    ########################
    def updateScreen(self):
        '''
        update the screen, called on every screen refresh
            This is where you implement drawing to the screen
        '''
        #CHANGEME - this is how you detect which segment, so that
        #           you can draw different things on different segments
        if self.state.currentSegment == 0: 
            # CHANGEME - this is how you get parameter settings
            dir = self.currentParams['dir']
            coherence = self.currentParams['coherence']
            # CHANGEME: here we ill draw the stimulus  
            self.rdk.display(direction=dir,coherence=coherence,speed=self.settings.config.speed)
        
        # draw a fixation cross
        self.pgl.fixationCross(color=self.state.fixationColor)
        
        # You do NOT call pglFlush, as this is done by the task code logic for you
        
    ########################
    # handleSubjectResponse
    ########################    
    def handleSubjectResponse(self, response, updateTime):
        '''
        Handle the subject response. Response will come in as an integer
        value of what button was pressed. The order of buttons is set
        in pgl.settings() in the field "responseKeys"
        '''
        # already received a response
        if self.state.gotResponse: return None
        # mark that we got a response
        self.state.gotResponse = True
        
        # check if response is correct 
        # CHANGEME, change to your logic
        if ((response==0 and self.currentParams['dir']==180) or
            (response==1 and self.currentParams['dir']==0)):
            correct = True 
            # set fixation cross to green
            self.state.fixationColor = [0,1,0]
        else:
            correct = False
            # set fixation cross to red
            self.state.fixationColor = [1,0,0]
            
        # return response type, if True/False then this will
        # display as green / red markers in the task summary plot
        # the value will be stored in
        return correct

    ########################
    # handleEvents
    ########################
    def handleEvents(self, events):
        '''
        handleEvents gets every keyboard/mouse/volume event passed
            to the experiment. Not typically needed to implement
        '''
        # CHANGEME - if you want to process events, you can do so
        #            with a loop here, event types are defined
        #            in pglExperiment and pglEvents. You do not
        #            typically need to use this callback
        for event in events: 
            pass


---

Annotated code for setting up experiment

---

In [ ]:
#closes any open pgl screens
pgl.cleanUp()

# setup experiment. 
#
# settingsName is one of the pgl.settings() that controls
# which screen to use and other settings. If it is ommitted will default
#  to whatever settings is set as default
#
# experimetName is the top level directory which the data will be stored in
#    data are stored as experimentName / subjectID / session_YYMMDD / run_HHMMSS
#
# subjectID is the ID of the subject, liek s0001
#
#e = pglExperiment(pgl,settingsName='CHANGEME',experimentName='CHANGEME', subjectID='CHANGEME')
#
# Here we will use default settings
e = pglExperiment(pgl)

# load the task you created
e.addTask(pgl_CHANGEME_Task(pgl))

---

Code for running experiment

---

In [ ]:
# starts up screen
e.initScreen()

# and runs experiment
e.run()

---

SCRATCH CODE BELOW

---

---

Code for extrating pRF bars (needs to be consolidated into its own tutorial)

---

In [ ]:
# hacked for now = grab task settings and create a pglBarTask with the same settings
t = next((task for task in e.tasks if task.settings.taskName == "Bar Mapping Task"), None)
barTask = pglBarTask(pgl)
barTask.settings = t.settings
barTask.data = t.data
barTask.state = t.state
print(barTask.settings.taskName)
frames = barTask.getStimulusFrames(e.data.events, e.settings)

In [ ]:
# display stimulus frames as a movie
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np

def showFramesJupyter(frames):
    """
    frames: numpy array with shape (numFrames, height, width, 4) - RGBA
    """
    
    numFrames = frames.shape[0]
    
    # Create output widget
    outputWidget = widgets.Output()
    
    # Create slider
    frameSlider = widgets.IntSlider(
        value=0,
        min=0,
        max=numFrames-1,
        description='Frame:',
        continuous_update=True,
        layout=widgets.Layout(width='400px')
    )
    
    # Create play button
    playButton = widgets.Play(
        value=0,
        min=0,
        max=numFrames-1,
        step=1,
        interval=100,
        description="Press play"
    )
    
    # Create increment/decrement buttons
    prevButton = widgets.Button(
        description='◀ Prev',
        button_style='',
        tooltip='Previous frame',
        icon='arrow-left'
    )
    
    nextButton = widgets.Button(
        description='Next ▶',
        button_style='',
        tooltip='Next frame',
        icon='arrow-right'
    )
    
    # Button click handlers
    def onPrevClick(b):
        if frameSlider.value > 0:
            frameSlider.value -= 1
    
    def onNextClick(b):
        if frameSlider.value < numFrames - 1:
            frameSlider.value += 1
    
    prevButton.on_click(onPrevClick)
    nextButton.on_click(onNextClick)
    
    # Link play button to slider
    widgets.jslink((playButton, 'value'), (frameSlider, 'value'))
    
    # Update function
    def updateFrame(change):
        with outputWidget:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(10, 7.5))
            ax.imshow(frames[change['new']])
            ax.set_title(f"Frame {change['new']} / {numFrames-1}")
            ax.axis('off')
            plt.tight_layout()
            plt.show()
            plt.close(fig)
    
    # Attach observer
    frameSlider.observe(updateFrame, names='value')
    
    # Layout widgets
    buttonBox = widgets.HBox([playButton, prevButton, nextButton])
    controlBox = widgets.VBox([buttonBox, frameSlider])
    
    display(controlBox)
    display(outputWidget)
    
    # Show first frame
    updateFrame({'new': 0})

# Usage
#showFramesJupyter(frames)
showFramesJupyter(stimImage)

In [ ]:
# format and save to .mat file (pRFStimImage.mat on Desktop)
import scipy.io
import numpy as np
import os

# Create copy and apply transformations
stimImage = frames.copy()

# Keep only first channel (R, drop GBA)
stimImage = stimImage[:, :, :, 0]

# Map values: 0.5 -> 0, everything else -> 1
stimImage = np.where(stimImage == 0.5, 0, 1)

# Flip left-right (horizontal flip)
stimImage = np.flip(stimImage, axis=2) 

# filepath
filepath = os.path.join(os.path.expanduser('~/Desktop'), 'pRFStimImage.mat')

# Save your frames array
scipy.io.savemat(filepath, { 'stimImage': stimImage })

In [ ]:
import inspect


instance = pglBarTask(pgl)

class_name = instance.__class__.__name__
module_name = instance.__class__.__module__
file_path = inspect.getfile(instance.__class__)

print(f"Class: {class_name}")
print(f"Module: {module_name}")
print(f"File: {file_path}")

In [ ]:
from pgl import pglActions
session = pglActions.loadSession.execute()

In [ ]:
session.runs[0].print()

In [ ]:
session.runs[0].data.print()